# 07 · Oferta: cuánto vende y cuánto gana cada restaurante

**Objetivo:** estimar las ventas anuales y la utilidad de cada restaurante del DENUE (2020 y 2025) y sumar la **oferta en pesos** de cada zona.

**Cómo se estima** (no hay ventas por establecimiento en datos públicos):

1. De los Censos Económicos 2024 (datos de 2023) se calcula, por **municipio × tipo de restaurante**:
   - **ventas por trabajador** = ingresos / personal ocupado
   - **margen** = (ingresos − gastos − sueldos) / ingresos
2. Del DENUE se toma el **tamaño** de cada restaurante (rango de personal ocupado → punto medio).
3. **Ventas estimadas** = ventas por trabajador × trabajadores. **Utilidad estimada** = ventas × margen.
4. Se **calibra** por municipio para que la suma de ventas estimadas de 2025 cuadre con el total de los Censos Económicos.

Todo queda en **pesos de 2023**. La "utilidad" es una aproximación antes de impuestos, depreciación e intereses.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import h3
import mapas

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
RES = 8
pesos = FuncFormatter(lambda v, _: f"${v:,.0f}")

## 1. Configuración

In [ ]:
TIPOS_RESTAURANTE = {
    "722511": "A la carta / comida corrida", "722512": "Mariscos", "722513": "Antojitos",
    "722514": "Tacos y tortas", "722515": "Cafeterías y neverías", "722516": "Autoservicio",
    "722517": "Pizzas, hamburguesas y pollos", "722518": "Otros para llevar",
    "722519": "Otros de consumo inmediato", "722411": "Bares y centros nocturnos",
    "722412": "Bares y centros nocturnos",
}
MUN_CVE = {"Corregidora": 6, "Huimilpan": 8, "El Marqués": 11, "Querétaro": 14}
EMPLEO_PUNTO_MEDIO = {"0 a 5 personas": 3, "6 a 10 personas": 8, "11 a 30 personas": 20, "31 a 50 personas": 40,
                      "51 a 100 personas": 75, "101 a 250 personas": 175, "251 y más personas": 300}
MILLONES = 1e6
MIN_UE_MUNICIPIO = 30   # con menos establecimientos, el margen y las ventas por trabajador del municipio son muy ruidosos: se usa el de la ZM

## 2. Parámetros por municipio y tipo (Censos Económicos)

Los bares y centros nocturnos (dos clases SCIAN) se suman antes de calcular las razones. Si un municipio tiene un tipo **oculto por confidencialidad** o con **menos de `MIN_UE_MUNICIPIO` establecimientos** (en celdas tan chicas un par de negocios mueve el margen a valores poco creíbles, como 61 %), se usa el valor de ese tipo en la **suma de los municipios de la ZM** con datos; si tampoco hay, el estatal y, al final, el del subsector 722 completo. Se prefiere la ZM al estado porque el estado incluye municipios con otra estructura (San Juan del Río, Tequisquiapan): por ejemplo, en antojitos el estado da unos $500 mil de ventas por trabajador y el municipio de Querétaro menos de $300 mil. La columna `fuente` dice cuál se usó.

In [ ]:
ce = pd.read_parquet(PROC / "ce2024_restaurantes.parquet")
VARS = ["ue", "h001a", "m000a", "k000a", "j000a"]

def razones(df):
    g = df.groupby(["cve_mun", "tipo"], dropna=False)[VARS].sum(min_count=1)
    g["ventas_por_trab"] = g["m000a"] * MILLONES / g["h001a"]
    g["ventas_por_ue"] = g["m000a"] * MILLONES / g["ue"]
    g["margen"] = (g["m000a"] - g["k000a"] - g["j000a"]) / g["m000a"]
    return g

clases = ce[ce["codigo"].isin(TIPOS_RESTAURANTE)].assign(tipo=lambda d: d["codigo"].map(TIPOS_RESTAURANTE))
sub = ce[ce["codigo"] == "722"].assign(tipo="(todo 722)")
r_clase, r_sub = razones(clases), razones(sub)
# Suma de los municipios de la ZM con datos publicados (cve_mun = 0): mejor respaldo que el estado, que incluye San Juan del Río, Tequisquiapan, etc.
r_zm = razones(clases[clases["cve_mun"].isin(MUN_CVE.values())].assign(cve_mun=0))

rc, rs, rz = r_clase.reset_index(), r_sub.reset_index(), r_zm.reset_index()

def buscar(tabla, mun, tipo):
    m = (tabla["tipo"] == tipo) & (tabla["cve_mun"].isna() if mun is None else tabla["cve_mun"] == mun)
    if mun:                                            # municipio: exige un mínimo de establecimientos
        m &= tabla["ue"] >= MIN_UE_MUNICIPIO
    f = tabla[m & np.isfinite(tabla["ventas_por_trab"]) & np.isfinite(tabla["margen"]) & (tabla["ventas_por_trab"] > 0)]
    return f.iloc[0] if len(f) else None

filas = []
for mun in MUN_CVE.values():
    for tipo in sorted(set(TIPOS_RESTAURANTE.values())):
        for fuente, tabla, m, t in (("municipio", rc, mun, tipo), ("ZM", rz, 0, tipo), ("estatal", rc, None, tipo),
                                    ("722 municipio", rs, mun, "(todo 722)"), ("722 estatal", rs, None, "(todo 722)")):
            f = buscar(tabla, m, t)
            if f is not None:
                filas.append({"cve_mun": mun, "tipo": tipo, "ventas_por_trab": f["ventas_por_trab"],
                              "ventas_por_ue": f["ventas_por_ue"], "margen": f["margen"], "fuente": fuente})
                break
param = pd.DataFrame(filas)
param.to_parquet(PROC / "ce2024_parametros_mun_tipo.parquet", index=False)
print(param["fuente"].value_counts().to_string())
param.pivot(index="tipo", columns="cve_mun", values="margen").round(2)

## 3. Gráfica · Ventas por trabajador y margen por tipo (municipio de Querétaro)

In [ ]:
q = param[param["cve_mun"] == 14].sort_values("ventas_por_trab")
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
axes[0].barh(q["tipo"], q["ventas_por_trab"], color="#2E6F95")
axes[0].xaxis.set_major_formatter(pesos); axes[0].set_title("Ventas anuales por trabajador", loc="left")
axes[1].barh(q["tipo"], q["margen"] * 100, color="#A8551A")
axes[1].set_title("Margen (% de las ventas)", loc="left")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Censos Económicos 2024 (datos de 2023) · municipio de Querétaro", x=0.01, ha="left")
plt.tight_layout(); plt.savefig(FIG / "07_ventas_margen_por_tipo.png", dpi=200, bbox_inches="tight"); plt.show()

## 4. Ventas y utilidad estimadas de cada restaurante

- **2020:** los 8,536 restaurantes que se siguen en el notebook 03 (su tamaño viene del DENUE 04/2020).
- **2025:** todos los restaurantes del DENUE 05/2025.

Los dos años se valúan con los parámetros de 2023, así que la diferencia entre años refleja cambios en número, tipo y tamaño de los restaurantes, no inflación.

In [ ]:
def a_hex(df):
    return [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(df["latitud"], df["longitud"])]

trans = pd.read_parquet(PROC / "transiciones_restaurantes_2020_2025.parquet")
den20 = pd.read_parquet(PROC / "denue_zmq_2020-04.parquet", columns=["id", "per_ocu"])
r20 = trans.merge(den20, on="id", how="left")
rest = pd.read_parquet(PROC / "restaurantes_zmq.parquet")
r25 = rest[rest["edicion"] == "2025-05"].copy()
r25["h3"] = a_hex(r25)

def valuar(df):
    df = df.copy()
    df["cve_mun"] = df["mun_nombre"].map(MUN_CVE)
    df["empleo"] = df["per_ocu"].map(EMPLEO_PUNTO_MEDIO).fillna(3)
    df = df.merge(param[["cve_mun", "tipo", "ventas_por_trab", "margen"]], on=["cve_mun", "tipo"], how="left")
    df["ventas_est"] = df["ventas_por_trab"] * df["empleo"]
    return df

r20, r25 = valuar(r20), valuar(r25)
print(f"Sin parámetro (revisar): 2020 = {r20['ventas_est'].isna().sum()} · 2025 = {r25['ventas_est'].isna().sum()}")

## 5. Calibración con el total de los Censos Económicos

Por municipio, se compara la suma de ventas estimadas de los restaurantes de 2025 con el total del subsector 722 en los Censos Económicos. El factor de ajuste lleva una a la otra. Un factor lejos de 1 indica que el punto medio del rango de personal no representa bien el tamaño real (o que el DENUE tiene restaurantes que el censo no). Se aplica el mismo factor a 2020. Si el total de un municipio está oculto por confidencialidad (Huimilpan), se usa el factor del resto de la ZM.

In [ ]:
tot_ce = rs[rs["cve_mun"].isin(MUN_CVE.values())].set_index("cve_mun")["m000a"] * MILLONES
cal = pd.DataFrame({"ventas_censo": tot_ce, "ventas_estimadas": r25.groupby("cve_mun")["ventas_est"].sum()})
cal["factor"] = cal["ventas_censo"] / cal["ventas_estimadas"]
ok = cal["factor"].notna()
FACTOR_ZM = cal.loc[ok, "ventas_censo"].sum() / cal.loc[ok, "ventas_estimadas"].sum()
cal["factor"] = cal["factor"].fillna(FACTOR_ZM)          # municipio con total oculto (Huimilpan): factor del resto de la ZM
print(f"Factor de la ZM (para municipios sin total en el censo): {FACTOR_ZM:.2f}")
for df in (r20, r25):
    df["ventas_est"] = df["ventas_est"] * df["cve_mun"].map(cal["factor"]).fillna(1.0)
    df["utilidad_est"] = df["ventas_est"] * df["margen"]
cal.rename_axis("cve_mun").reset_index().to_parquet(PROC / "ce2024_calibracion_mun.parquet", index=False)
cal.assign(municipio=lambda d: d.index.map({v: k for k, v in MUN_CVE.items()})).round(2)

In [ ]:
resumen = pd.DataFrame({
    "restaurantes": [len(r20), len(r25)],
    "ventas totales (millones)": [r20["ventas_est"].sum() / MILLONES, r25["ventas_est"].sum() / MILLONES],
    "ventas medianas por restaurante": [r20["ventas_est"].median(), r25["ventas_est"].median()],
    "utilidad mediana por restaurante": [r20["utilidad_est"].median(), r25["utilidad_est"].median()],
}, index=["2020", "2025"])
resumen.round(0)

## 6. Oferta por hexágono

Ventas estimadas de todos los restaurantes del hexágono (total y por tipo), y la misma suma en el **entorno** (hexágono + 6 vecinos), que es el área de la que compite cada restaurante.

In [ ]:
def oferta_hex(df, sufijo):
    tot = df.groupby("h3")["ventas_est"].sum().rename(f"oferta_{sufijo}")
    por_tipo = df.pivot_table(index="h3", columns="tipo", values="ventas_est", aggfunc="sum", fill_value=0)
    por_tipo.columns = [f"oferta_{sufijo}__{t}" for t in por_tipo.columns]
    return pd.concat([tot, por_tipo], axis=1)

def entorno(tabla):
    idx = set(tabla.index)
    filas = {}
    for c in tabla.index:
        vec = [v for v in h3.grid_disk(c, 1) if v in idx]
        filas[c] = tabla.loc[vec].sum()
    return pd.DataFrame(filas).T.add_suffix("_entorno")

of20, of25 = oferta_hex(r20, "2020"), oferta_hex(r25, "2025")
oferta = of20.join(of25, how="outer").fillna(0)
oferta = oferta.join(entorno(oferta))
oferta.rename_axis("h3").reset_index().to_parquet(PROC / f"oferta_hex{RES}.parquet", index=False)
r20[["id", "tipo", "h3", "cve_mun", "empleo", "ventas_est", "utilidad_est"]].to_parquet(PROC / "restaurantes_ingresos_2020.parquet", index=False)
r25[["id", "tipo", "h3", "cve_mun", "empleo", "ventas_est", "utilidad_est"]].to_parquet(PROC / "restaurantes_ingresos_2025.parquet", index=False)
print(f"Oferta por hexágono: {len(oferta):,} hexágonos → oferta_hex{RES}.parquet")

## 7. Mapa · Ventas estimadas de restaurantes por hexágono, 2025

In [ ]:
v = oferta["oferta_2025"]
v = v[v > 0] / MILLONES
BORDES = [0, 1, 5, 20, 50, 150, max(151, v.max() + 1)]
ETIQ = ["< 1", "1–5", "5–20", "20–50", "50–150", "150 o más"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#FFF3D6", "#FDD49E", "#FDAE6B", "#F16913", "#C2410C", "#7F2704"])
fig, ax, polys = mapas.base_mapa(v.index)
coll = mapas.capa_hex(ax, polys, array=v.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Ventas anuales estimadas (millones de pesos de 2023)")
ax.set_title("Oferta de restaurantes en pesos · 2025", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "07_mapa_oferta.png", dpi=200, bbox_inches="tight"); plt.show()